In [1]:
import spacy
!python -m spacy download en_core_web_lg
nlp= spacy.load("en_core_web_lg")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
doc =nlp("bread sandwich burger car tiger human wheat")

In [3]:
for token in doc:
  print(token.text, "Vector: ", token.has_vector,"OOV: ", token.is_oov)

bread Vector:  True OOV:  False
sandwich Vector:  True OOV:  False
burger Vector:  True OOV:  False
car Vector:  True OOV:  False
tiger Vector:  True OOV:  False
human Vector:  True OOV:  False
wheat Vector:  True OOV:  False


In [4]:
doc[0].vector.shape

(300,)

In [5]:
doc1= nlp("bread")
doc1.vector.shape

(300,)

In [6]:
for token in doc:
  print(f"{token.text} --> {doc1.text} :", token.similarity(doc1))

bread --> bread : 1.0
sandwich --> bread : 0.6874560117721558
burger --> bread : 0.544037401676178
car --> bread : 0.16441145539283752
tiger --> bread : 0.14492353796958923
human --> bread : 0.21103660762310028
wheat --> bread : 0.6572456359863281


In [7]:
def similarity(base_word, words_to_compare):
  base= nlp(base_word)
  doc= nlp(words_to_compare)
  for token in doc:
    print(f"{token.text} --> {base.text} :", token.similarity(base))

In [8]:
similarity("iphone", "apple samsung iphone dog kitten")

apple --> iphone : 0.6339781284332275
samsung --> iphone : 0.6678677797317505
iphone --> iphone : 0.9999999403953552
dog --> iphone : 0.1743103712797165
kitten --> iphone : 0.14685814082622528


In [9]:
king= nlp.vocab['king'].vector
man= nlp.vocab['man'].vector
woman= nlp.vocab['woman'].vector
queen= nlp.vocab['queen'].vector
result= king-man+woman

In [10]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity([result], [queen])

array([[0.7880844]], dtype=float32)

In [13]:
import pandas as pd
df= pd.read_csv("/content/WELFake_Dataset.csv")
df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [19]:
df=df.drop(['Unnamed: 0', 'title'], axis=1)

In [20]:
df.shape

(72134, 2)

In [21]:
df.label.value_counts()

,count
label,
1,37106
0,35028


In [22]:
df.isna().any()

,0
text,True
label,False


In [26]:
df=df.head(2000)

In [27]:
df['text_vector']= df['text'].apply(lambda x: nlp(x).vector)

/tmp/ipykernel_426/3337290239.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text_vector']= df['text'].apply(lambda x: nlp(x).vector)


In [28]:
df.head()

,text,label,text_vector
0,No comment is expected from Barack Obama Membe...,1,"[-0.057963405, 0.15719564, -0.115160584, -0.03..."
1,Did they post their votes for Hillary already?,1,"[-0.17601645, 0.16607201, 0.0054855505, -0.053..."
2,"Now, most of the demonstrators gathered last ...",1,"[0.011617082, 0.052363843, -0.1188232, 0.01947..."
3,A dozen politically active pastors came here f...,0,"[-0.00069660903, 0.14964917, -0.06404743, -0.0..."
4,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1,"[-0.03320415, 0.09715335, -0.027121484, -0.021..."


In [29]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(
    df.text_vector.values,
    df.label,
    test_size=0.2,
    random_state=2022
)

In [32]:
type(X_train)

numpy.ndarray

In [33]:
import numpy as np
X_train_2d= np.stack(X_train)

In [36]:
print(type(X_train_2d))
X_train_2d.shape

<class 'numpy.ndarray'>


(1600, 300)

In [38]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
mn= MinMaxScaler()
X_train_2d= mn.fit_transform(X_train_2d)
model= MultinomialNB()
model.fit(X_train_2d, y_train)

MultinomialNB()

In [39]:
X_test= np.stack(X_test)
X_test= mn.transform(X_test)
model.score(X_test, y_test)

0.775

In [41]:
from sklearn.metrics import classification_report
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.80      0.69      0.74       188
           1       0.76      0.85      0.80       212

    accuracy                           0.78       400
   macro avg       0.78      0.77      0.77       400
weighted avg       0.78      0.78      0.77       400

